# Test Repository Data Dates

This notebook tests the date ranges for all data types available in the repository tool.
Used to validate simulation cutoff approach - we need to know what the earliest/latest dates are for each data type.

In [1]:
import sys
sys.path.append('..')

from datetime import datetime, timedelta
from app.repositories.ratings_data import (
    get_stock_grades_individual, 
    get_stock_grades_summary, 
    get_ratings, 
    get_analyst_recommendations, 
    get_price_target_summary
)
from app.repositories.news_data import (
    get_press_releases, 
    get_stock_news, 
    get_price_target_news
)
from app.repositories.transcripts_data import (
    get_earnings_transcripts, 
    get_latest_transcript
)
from app.repositories.price_data import get_dividends_series
from app.repositories.etf_data import get_etf_info, get_etf_holdings

import pandas as pd
import yaml

## Test Configuration

We'll test with a few common consumer staples tickers

In [2]:
# Test tickers from consumer staples sector
test_tickers = ['KO', 'PG', 'WMT', 'COST']

# Date ranges for testing
now = datetime.now()
start_date = now - timedelta(days=365 * 2)  # 2 years back
end_date = now

print(f"Testing period: {start_date.date()} to {end_date.date()}")

Testing period: 2023-10-01 to 2025-09-30


## Helper Function to Extract Dates

In [3]:
def extract_dates_from_data(data, date_field='date'):
    """Extract min and max dates from data structure."""
    if not data:
        return None, None
    
    dates = []
    
    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict) and date_field in item:
                dates.append(item[date_field])
    elif isinstance(data, dict):
        if date_field in data:
            dates.append(data[date_field])
        # Check nested items
        if 'items' in data and isinstance(data['items'], list):
            for item in data['items']:
                if isinstance(item, dict) and date_field in item:
                    dates.append(item[date_field])
    
    if not dates:
        return None, None
    
    # Convert to datetime if string
    parsed_dates = []
    for d in dates:
        if isinstance(d, str):
            try:
                parsed_dates.append(pd.to_datetime(d))
            except:
                pass
        elif isinstance(d, datetime):
            parsed_dates.append(d)
    
    if not parsed_dates:
        return None, None
    
    return min(parsed_dates), max(parsed_dates)

## Test 1: Ratings Data

In [4]:
print("\n" + "="*80)
print("RATINGS DATA TESTS")
print("="*80)

ratings_results = []

for ticker in test_tickers:
    print(f"\nTesting {ticker}...")
    
    try:
        # Stock grades individual
        data = get_stock_grades_individual(ticker, start=start_date, end=end_date)
        min_date, max_date = extract_dates_from_data(data)
        ratings_results.append({
            'ticker': ticker,
            'data_type': 'grades_individual',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Ratings
        data = get_ratings(ticker, start=start_date, end=end_date)
        min_date, max_date = extract_dates_from_data(data)
        ratings_results.append({
            'ticker': ticker,
            'data_type': 'ratings',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Analyst recommendations
        data = get_analyst_recommendations(ticker, start=start_date, end=end_date)
        min_date, max_date = extract_dates_from_data(data)
        ratings_results.append({
            'ticker': ticker,
            'data_type': 'analyst_recommendations',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Price target summary
        data = get_price_target_summary(ticker)
        min_date, max_date = extract_dates_from_data(data, 'lastUpdated')
        ratings_results.append({
            'ticker': ticker,
            'data_type': 'price_target_summary',
            'count': 1 if data else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
    except Exception as e:
        print(f"Error with {ticker}: {e}")

ratings_df = pd.DataFrame(ratings_results)
print("\n", ratings_df)


RATINGS DATA TESTS

Testing KO...

Testing PG...

Testing WMT...

Testing COST...

    ticker                data_type  count   min_date   max_date
0      KO        grades_individual      0 2023-10-10 2025-09-11
1      KO                  ratings      0 2023-10-02 2025-09-16
2      KO  analyst_recommendations      0 2025-07-08 2025-09-16
3      KO     price_target_summary      1        NaT        NaT
4      PG        grades_individual      0 2023-10-11 2025-08-15
5      PG                  ratings      0 2023-10-02 2025-09-16
6      PG  analyst_recommendations      0 2025-07-08 2025-09-16
7      PG     price_target_summary      1        NaT        NaT
8     WMT        grades_individual      0 2023-10-25 2025-09-03
9     WMT                  ratings      0 2023-10-02 2025-09-16
10    WMT  analyst_recommendations      0 2025-07-08 2025-09-16
11    WMT     price_target_summary      1        NaT        NaT
12   COST        grades_individual      0 2023-10-02 2025-08-27
13   COST          

## Test 2: News Data

In [5]:
print("\n" + "="*80)
print("NEWS DATA TESTS")
print("="*80)

news_results = []

for ticker in test_tickers:
    print(f"\nTesting {ticker}...")
    
    try:
        # Press releases
        data = get_press_releases(ticker, start=start_date, end=end_date, limit=50, ascending=False)
        min_date, max_date = extract_dates_from_data(data)
        news_results.append({
            'ticker': ticker,
            'data_type': 'press_releases',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Stock news
        data = get_stock_news(ticker, start=start_date, end=end_date, limit=50, ascending=False)
        min_date, max_date = extract_dates_from_data(data, 'publishedDate')
        news_results.append({
            'ticker': ticker,
            'data_type': 'stock_news',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Price target news
        data = get_price_target_news(ticker, start=start_date, end=end_date, limit=50, ascending=False)
        min_date, max_date = extract_dates_from_data(data, 'publishedDate')
        news_results.append({
            'ticker': ticker,
            'data_type': 'price_target_news',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
    except Exception as e:
        print(f"Error with {ticker}: {e}")

news_df = pd.DataFrame(news_results)
print("\n", news_df)


NEWS DATA TESTS

Testing KO...

Testing PG...

Testing WMT...

Testing COST...

    ticker          data_type  count            min_date            max_date
0      KO     press_releases      0                 NaT                 NaT
1      KO         stock_news      0 2025-08-20 05:21:00 2025-09-16 16:21:41
2      KO  price_target_news      0 2023-11-21 08:25:00 2025-07-21 13:54:00
3      PG     press_releases      0                 NaT                 NaT
4      PG         stock_news      0 2025-07-29 07:23:00 2025-09-11 12:46:33
5      PG  price_target_news      0 2023-10-18 02:56:00 2025-07-31 09:24:50
6     WMT     press_releases      0                 NaT                 NaT
7     WMT         stock_news      0 2025-08-27 06:01:00 2025-09-16 10:53:00
8     WMT  price_target_news      0 2024-05-17 05:17:00 2025-09-03 13:49:03
9    COST     press_releases      0                 NaT                 NaT
10   COST         stock_news      0 2025-08-13 09:17:54 2025-09-16 14:42:46
11   C

## Test 3: Transcripts Data

In [6]:
print("\n" + "="*80)
print("TRANSCRIPTS DATA TESTS")
print("="*80)

transcripts_results = []

for ticker in test_tickers:
    print(f"\nTesting {ticker}...")
    
    try:
        # Earnings transcripts
        data = get_earnings_transcripts(ticker, start_year=now.year - 2, end_year=now.year, limit=10)
        min_date, max_date = extract_dates_from_data(data)
        transcripts_results.append({
            'ticker': ticker,
            'data_type': 'earnings_transcripts',
            'count': len(data) if isinstance(data, list) else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
        # Latest transcript
        data = get_latest_transcript(ticker)
        min_date, max_date = extract_dates_from_data(data)
        transcripts_results.append({
            'ticker': ticker,
            'data_type': 'latest_transcript',
            'count': 1 if data else 0,
            'min_date': min_date,
            'max_date': max_date
        })
        
    except Exception as e:
        print(f"Error with {ticker}: {e}")

transcripts_df = pd.DataFrame(transcripts_results)
print("\n", transcripts_df)


TRANSCRIPTS DATA TESTS

Testing KO...

Testing PG...

Testing WMT...

Testing COST...

   ticker             data_type  count   min_date   max_date
0     KO  earnings_transcripts      0 2023-04-24 2025-04-29
1     KO     latest_transcript      1 2025-04-29 2025-04-29
2     PG  earnings_transcripts      0 2023-01-19 2025-04-24
3     PG     latest_transcript      1 2025-04-24 2025-04-24
4    WMT  earnings_transcripts      0 2022-11-15 2025-02-20
5    WMT     latest_transcript      1 2025-02-20 2025-02-20
6   COST  earnings_transcripts      0 2023-03-02 2025-05-29
7   COST     latest_transcript      1 2025-05-29 2025-05-29


## Test 4: Dividends Data

In [7]:
print("\n" + "="*80)
print("DIVIDENDS DATA TESTS")
print("="*80)

dividends_results = []

for ticker in test_tickers:
    print(f"\nTesting {ticker}...")
    
    try:
        # Dividends series
        series = get_dividends_series(ticker, start_date, end_date)
        
        if not series.empty:
            min_date = series.index.min()
            max_date = series.index.max()
        else:
            min_date = None
            max_date = None
        
        dividends_results.append({
            'ticker': ticker,
            'data_type': 'dividends_series',
            'count': len(series),
            'min_date': min_date,
            'max_date': max_date
        })
        
    except Exception as e:
        print(f"Error with {ticker}: {e}")

dividends_df = pd.DataFrame(dividends_results)
print("\n", dividends_df)


DIVIDENDS DATA TESTS

Testing KO...

Testing PG...

Testing WMT...

Testing COST...

   ticker         data_type  count   min_date   max_date
0     KO  dividends_series      8 2023-11-30 2025-09-15
1     PG  dividends_series      8 2023-10-19 2025-07-18
2    WMT  dividends_series      8 2023-12-07 2025-08-15
3   COST  dividends_series      9 2023-11-02 2025-08-01


## Summary: All Data Types

In [8]:
print("\n" + "="*80)
print("SUMMARY - ALL DATA TYPES")
print("="*80)

# Combine all results
all_results = pd.concat([
    ratings_df,
    news_df,
    transcripts_df,
    dividends_df
], ignore_index=True)

print("\nComplete Results:")
print(all_results.to_string())

# Group by data type to see overall date ranges
print("\n" + "="*80)
print("DATE RANGES BY DATA TYPE")
print("="*80)

summary = all_results.groupby('data_type').agg({
    'min_date': 'min',
    'max_date': 'max',
    'count': 'sum'
}).reset_index()

print("\n", summary.to_string())


SUMMARY - ALL DATA TYPES

Complete Results:
   ticker                data_type  count            min_date            max_date
0      KO        grades_individual      0 2023-10-10 00:00:00 2025-09-11 00:00:00
1      KO                  ratings      0 2023-10-02 00:00:00 2025-09-16 00:00:00
2      KO  analyst_recommendations      0 2025-07-08 00:00:00 2025-09-16 00:00:00
3      KO     price_target_summary      1                 NaT                 NaT
4      PG        grades_individual      0 2023-10-11 00:00:00 2025-08-15 00:00:00
5      PG                  ratings      0 2023-10-02 00:00:00 2025-09-16 00:00:00
6      PG  analyst_recommendations      0 2025-07-08 00:00:00 2025-09-16 00:00:00
7      PG     price_target_summary      1                 NaT                 NaT
8     WMT        grades_individual      0 2023-10-25 00:00:00 2025-09-03 00:00:00
9     WMT                  ratings      0 2023-10-02 00:00:00 2025-09-16 00:00:00
10    WMT  analyst_recommendations      0 2025-07-08 

## Test: Validate September 2024 Cutoff

This tests whether we have sufficient data before September 2024 for simulation

In [9]:
print("\n" + "="*80)
print("SEPTEMBER 2024 CUTOFF VALIDATION")
print("="*80)

cutoff = pd.to_datetime('2024-09-30')

print(f"\nSimulation Cutoff: {cutoff.date()}")
print(f"\nData Types with data BEFORE cutoff:")

for _, row in summary.iterrows():
    data_type = row['data_type']
    min_date = row['min_date']
    max_date = row['max_date']
    
    if pd.notna(min_date) and pd.notna(max_date):
        has_data_before = min_date < cutoff
        status = "✓ GOOD" if has_data_before else "✗ NO DATA"
        print(f"{data_type:30} {status:10} (earliest: {min_date.date()})")
    else:
        print(f"{data_type:30} {'✗ NO DATA':10}")


SEPTEMBER 2024 CUTOFF VALIDATION

Simulation Cutoff: 2024-09-30

Data Types with data BEFORE cutoff:
analyst_recommendations        ✗ NO DATA  (earliest: 2025-07-08)
dividends_series               ✓ GOOD     (earliest: 2023-10-19)
earnings_transcripts           ✓ GOOD     (earliest: 2022-11-15)
grades_individual              ✓ GOOD     (earliest: 2023-10-02)
latest_transcript              ✗ NO DATA  (earliest: 2025-02-20)
press_releases                 ✗ NO DATA 
price_target_news              ✓ GOOD     (earliest: 2023-10-18)
price_target_summary           ✗ NO DATA 
ratings                        ✓ GOOD     (earliest: 2023-10-02)
stock_news                     ✗ NO DATA  (earliest: 2025-07-29)
